Importations & configuration

In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../env"))
sys.path.append(os.path.abspath("../pricing_agent"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from market import TicketMarketModel

try:
    plt.style.use("seaborn-v0_8-darkgrid")
except OSError:
    plt.style.use("ggplot")

In [ ]:
# --- Une vente complète sous tarification fixe -------------------------------
ARTIST = "Powerwolf"      # None -> artiste tiré au sort (randomisation de domaine)
VENUE = "Komplex 457"     # None -> salle tirée au sort
FIXED_PRICE = 55.0
HORIZON = 120
SEED = 1

market = TicketMarketModel(artist=ARTIST, venue=VENUE, horizon=HORIZON, seed=SEED)
market.current_price = FIXED_PRICE

print(f"{market.artist.name} ({market.genre.name}) @ {market.venue.name}")
print(f"  popularité   : {market.popularity_index:.1f}/10")
print(f"  capacité     : {market.initial_tickets} places | Fan Cost Index {market.fan_cost_index:.0f} CHF")
print(f"  acheteurs pot.: {market.n_potential_buyers}")
print(f"  prix fixe    : {FIXED_PRICE:.0f} CHF")

rows = []
while not market.done:
    market.step()
    active = sum(1 for b in market.buyers if b.visits_made > 0 and not b.has_bought)
    rows.append(
        {
            "t_ecoule": market.initial_time - market.time_remaining,
            "visites_baseline": market.expected_visits_by_step[market._step_index - 1],
            "visites_reelles": market.visits_this_step,
            "ventes_tour": market.tickets_sold_this_step,
            "vendus_cumul": market.tickets_sold_total,
            "ca_cumul": market.tickets_sold_total * FIXED_PRICE,
            "actifs_sans_billet": active,
        }
    )

df_results = pd.DataFrame(rows)
print(
    f"\nVendus : {market.tickets_sold_total}/{market.initial_tickets} "
    f"({market.fill_rate:.1%}) | CA billetterie : {market.tickets_sold_total * FIXED_PRICE:,.0f} CHF"
)
df_results.tail()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle(
    f"Vente sous prix fixe — {market.artist.name} @ {market.venue.name} "
    f"(popularité {market.popularity_index:.1f}/10, {FIXED_PRICE:.0f} CHF)",
    fontsize=14, fontweight="bold",
)
x = df_results["t_ecoule"]

axes[0, 0].plot(x, df_results["vendus_cumul"], color="tab:blue", lw=2)
axes[0, 0].axhline(market.initial_tickets, color="red", ls="--", label="Capacité")
axes[0, 0].set(title="Billets vendus (cumulé)", xlabel="Temps écoulé", ylabel="Billets")
axes[0, 0].legend()

axes[0, 1].bar(x, df_results["visites_reelles"], color="tab:orange", alpha=0.6, label="Visites réalisées")
axes[0, 1].plot(x, df_results["visites_baseline"], color="tab:brown", lw=1.0, ls="--", label="Intensité baseline (prix neutre)")
axes[0, 1].plot(x, df_results["ventes_tour"], color="tab:purple", lw=1.5, label="Ventes du tour")
axes[0, 1].set(title="Trafic plateforme (Poisson non homogène, intensité endogène)", xlabel="Temps écoulé")
axes[0, 1].legend()

axes[1, 0].plot(x, df_results["ca_cumul"], color="tab:green", lw=2)
axes[1, 0].set(title="Chiffre d'affaires billetterie (cumulé)", xlabel="Temps écoulé", ylabel="CHF")

axes[1, 1].plot(x, df_results["actifs_sans_billet"], color="tab:red", lw=2)
axes[1, 1].set(title="Demande non servie (agents actifs sans billet)", xlabel="Temps écoulé", ylabel="Agents")

plt.tight_layout()
plt.show()